**AI Tool Used:** ChatGPT
**Purpose:** I used it to understand some parts of the project, like how MediaPipe landmarks work and how to connect Vision → Text → Audio. The final code, implementation, and explanations were written by me after testing and modifying everything myself.

# Midterm Project – Sign Language Multimodal System
**Name:** Abdullah Hani Abdellatif Al-Shobaki  
**Student Number:** 2284612     
**Course:** AIN4311 – Special Topics in Artifical Intelligence Engineering I  
**Due Date:** December 17, 2025

In [2]:
!python3 -m pip install mediapipe opencv-python gTTS transformers scikit-learn torch torchvision torchaudio pandas numpy jupyter --quiet

In [3]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from gtts import gTTS
from IPython.display import Audio, display

In [4]:
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

def get_landmarks(img, hands):
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    res = hands.process(rgb)
    if res.multi_hand_landmarks:
        lm = res.multi_hand_landmarks[0]
        pts = []
        for p in lm.landmark:
            pts.extend([p.x, p.y, p.z])
        return pts
    return None

In [5]:
def collect_data(label, n=40):
    cap = cv2.VideoCapture(0)
    collected = []

    with mp_hands.Hands(max_num_hands=1) as hands:
        count = 0
        while count < n:
            ok, frame = cap.read()
            if not ok:
                break

            pts = get_landmarks(frame, hands)
            txt = f"{label}  {count}/{n}"
            cv2.putText(frame, txt, (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1,(0,255,0),2)

            cv2.imshow("collect", frame)
            if cv2.waitKey(1) & 255 == ord('q'):
                break

            if pts is not None:
                collected.append(pts + [label])
                count += 1

    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)
    return collected

In [5]:
data_all = []

data_all += collect_data("HELLO", 40)
data_all += collect_data("YES", 40)
data_all += collect_data("NO", 40)
data_all += collect_data("THANK YOU", 40)

df = pd.DataFrame(data_all)
df.to_csv("gesture_data.csv", index=False)
df.head()

I0000 00:00:1764863458.661943   41163 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1764863458.669919   43963 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1764863458.676181   43963 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1764863459.687243   43964 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
I0000 00:00:1764863471.339438   41163 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
W0000 00:00:1764863471.349185   44263 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference.

,0,1,2,3,4,5,6,7,8,9,...,54,55,56,57,58,59,60,61,62,63
0,0.333665,0.839954,-2.801744e-07,0.342170,0.741694,-0.043247,0.346709,0.671587,-0.073358,0.321482,...,0.400292,0.981361,-0.086418,0.428202,0.976902,-0.080511,0.449361,0.973422,-0.074130,HELLO
1,0.370844,0.810821,1.601962e-07,0.359950,0.737134,-0.074904,0.361171,0.691527,-0.128951,0.349678,...,0.484388,0.938929,-0.130748,0.519807,0.926251,-0.142124,0.547311,0.911182,-0.148424,HELLO
2,0.424847,0.818496,1.012422e-07,0.404095,0.744268,-0.071708,0.391698,0.699410,-0.123073,0.374701,...,0.527485,0.949833,-0.106991,0.560828,0.937589,-0.118680,0.588841,0.923976,-0.126537,HELLO
3,0.440136,0.824215,-1.616217e-07,0.412795,0.758938,-0.059036,0.396743,0.722020,-0.104328,0.385258,...,0.532521,0.963889,-0.101372,0.566563,0.953819,-0.110405,0.594910,0.951419,-0.116154,HELLO
4,0.452467,0.817296,-2.742056e-07,0.426558,0.749369,-0.058317,0.411700,0.715733,-0.100898,0.397393,...,0.545216,0.958344,-0.099409,0.580290,0.949914,-0.109096,0.608057,0.951781,-0.115074,HELLO


## Dataset Summary
- **Total Samples**: 160 (40 per gesture)
- **Gestures**: HELLO, YES, NO, THANK YOU
- **Features**: 63 (21 landmarks × 3 coordinates)
- **Train/Test Split**: 80/20

In [6]:
df = pd.read_csv("gesture_data.csv")
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

mlp = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500)
mlp.fit(X_train, y_train)

preds = mlp.predict(X_test)
print("accuracy:", accuracy_score(y_test, preds))

accuracy: 0.96875


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [7]:
tok = AutoTokenizer.from_pretrained("google/flan-t5-large")
t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")

def make_sentence(label):
    word = label.lower()
    prompt = f"""
    You write one short, natural English sentence that includes the given gesture word.
    You always use the real meaning of the gesture, not a random usage.
    The sentence must be simple, human, and realistic.

    Gesture meanings:
    hello = greeting someone
    yes = agreeing
    no = refusing / declining
    thank you = expressing gratitude

    Rules:
    - Use the gesture word exactly once.
    - Make the sentence 6–12 words.
    - No lists. No explanations. No repeating the word.
    - Output ONLY the sentence. Nothing else.

    Gesture word: "{word}"
    Sentence:
    """.strip()

    inputs = tok(prompt, return_tensors="pt")
    out = t5.generate(**inputs, max_length=25)
    text = tok.decode(out[0], skip_special_tokens=True).strip()
    return text

In [8]:
def speak(text):
    tts = gTTS(text)
    tts.save("speech.mp3")
    display(Audio("speech.mp3", autoplay=True))

## Real-Time Recognition Instructions
- The webcam window will open
- Show your hand gestures to the camera
- Press **'s'** to speak the current detected gesture
- Press **'q'** to quit

In [10]:
cap = cv2.VideoCapture(0)
strawberry = "Sign Language Recognition"

last_gesture = None

if not cap.isOpened():
    print("Error: Could not open the camera")
else:
    with mp_hands.Hands(max_num_hands=1) as hands:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            pts = get_landmarks(frame, hands)

            if pts is not None:
                arr = np.array(pts).reshape(1, -1)
                try:
                    g = mlp.predict(arr)[0]
                    last_gesture = g
                    cv2.putText(frame, g, (10, 40),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                except:
                    pass

            cv2.imshow(strawberry, frame)
            k = cv2.waitKey(1) & 255

            if k == ord('s'):
                if last_gesture:
                    try:
                        banana = make_sentence(last_gesture)
                        print("Sentence:", banana)
                        speak(banana)
                    except:
                        pass

            if k == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)
    print("Finished. How was it?")

I0000 00:00:1765142749.456913  412709 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
W0000 00:00:1765142749.463674  418611 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1765142749.473826  418611 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Sentence: The man said hello to the woman.


Sentence: I said yes to the idea.


Sentence: I'm not going to do that.


Sentence: Thank you for the gift.


Finished. How was it?
